# AI-Based Botnet Traffic Detection
**Course:** Computer Networks | **Faculty:** Ms. Pragya Vats | Karnavati University (UIT)

This notebook walks through the full pipeline: **load → clean → split → train → evaluate → visualise**.

A *botnet* is a group of infected devices (bots) controlled by an attacker through a command-and-control (C&C) server.
Bot traffic often looks different from normal traffic at the flow level: small packets, regular (periodic) timing, and unusual ports/durations.
We train machine-learning models on flow features to classify each flow as **Normal** or **Suspicious (Bot)**.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")          # so we can import our pipeline module
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import botnet_detection as bd

pd.set_option("display.max_columns", 30)
%matplotlib inline

## 1. Load the dataset
Download **CIC-IDS2017 (MachineLearningCVE)** from the Canadian Institute for Cybersecurity and put the CSV files in `data/`.
The botnet traffic ("Bot") is in `Friday-WorkingHours-Morning.pcap_ISCX.csv`.

In [ ]:
DATA_PATH = "../data/Friday-WorkingHours-Morning.pcap_ISCX.csv"   # file or folder
df, files_used = bd.load_dataset(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
# Label distribution BEFORE cleaning
df["Label"].str.strip().value_counts()

## 2. Clean the data
- keep only `BENIGN` and `Bot` flows (binary task: normal vs botnet)
- replace `inf` / `NaN` values (e.g. flow rates when duration is 0) and drop those rows
- drop duplicate rows **before** splitting, to avoid the same flow appearing in both train and test (data leakage)
- drop constant columns (no information)

In [ ]:
X, y, report = bd.clean_dataset(df)
report

In [ ]:
bd.plot_class_distribution(y)
plt.show()

**Observation:** the Bot class is very small compared to Normal traffic (a *class-imbalanced* problem).
Accuracy alone would be misleading — a model that always says "Normal" would still score ~99%.
So we focus on **precision, recall and F1-score** for the Bot class.

## 3. Train / test split
We keep 70% of the data for training and hold out 30% for testing. The split is *stratified*, so both parts keep the same Bot : Normal ratio.

In [ ]:
X_train, X_test, y_train, y_test = bd.split_data(X, y, test_size=0.3)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print("Bot flows in train:", y_train.sum(), "| in test:", y_test.sum())

## 4. Train the models
- **Decision Tree** — simple baseline
- **Random Forest** — main model: many decision trees voting together
- **Logistic Regression** — linear baseline (with feature scaling)

All use `class_weight="balanced"` so mistakes on the rare Bot class cost more.

In [ ]:
models = bd.train_models(bd.build_models(), X_train, y_train)

## 5. Evaluate on unseen test data
- **Precision:** of the flows flagged as Bot, how many really were Bot? (low precision = many false alarms)
- **Recall:** of all real Bot flows, how many did we catch? (low recall = missed botnet traffic)
- **F1-score:** balance of precision and recall
- **False positive rate:** share of normal flows wrongly flagged

In [ ]:
metrics = bd.evaluate_all(models, X_test, y_test)
metrics.round(4)

In [ ]:
print(bd.text_reports(models, X_test, y_test))

In [ ]:
bd.plot_confusion_matrices(models, X_test, y_test)
plt.show()

In [ ]:
bd.plot_metric_comparison(metrics)
plt.show()

In [ ]:
bd.plot_pr_curves(models, X_test, y_test)
plt.show()

## 6. Which features matter? (explainability)
Random Forest tells us which flow features helped most to separate Bot from Normal traffic.
Link these back to networking concepts in your report (e.g. inter-arrival times → beaconing, packet lengths → small C&C messages).

In [ ]:
bd.plot_feature_importance(models["Random Forest"], list(X.columns))
plt.show()

## 7. Try the model on a few flows (Normal / Suspicious)

In [ ]:
sample = X_test.sample(10, random_state=1)
preds = bd.predict_flows(models["Random Forest"], list(X.columns), sample)
preds["Actual"] = y_test.loc[sample.index].map({0: "Normal", 1: "Suspicious"})
preds[["Bot_probability", "Prediction", "Actual"]]

## 8. Save everything for the report, PPT and demo app

In [ ]:
out = "../results"
bd.plot_class_distribution(y, out)
bd.plot_confusion_matrices(models, X_test, y_test, out)
bd.plot_metric_comparison(metrics, out)
bd.plot_feature_importance(models["Random Forest"], list(X.columns), out_dir=out)
bd.plot_pr_curves(models, X_test, y_test, out)
bd.save_artifacts(models, metrics, bd.text_reports(models, X_test, y_test),
                  report, files_used, list(X.columns),
                  results_dir=out, models_dir="../models")
print("Saved to results/ and models/")

## 9. Conclusion
Write 3-4 lines here **using your own measured numbers**: which model performed best, its precision / recall / F1 on the Bot class,
the false-positive rate, and the main limitation (imbalanced data, one dataset, one botnet family).